# Insurance Submission Intake Assistant — FAQ Notebook

This notebook demonstrates a small, **offline** FAQ assistant using the project's insurance FAQ dataset. It finds a close FAQ question and returns the answer already written in that dataset.

## Important boundaries

- The embedded FAQ JSON is the notebook's only business knowledge source.
- This notebook does not connect to production systems or live submission data.
- It does not make coverage or underwriting decisions.
- It does not use real customer, broker, policy, or submission information.
- Offline matching is not an LLM call. No provider, API, or credential is required for the demonstration.

## 1. Prepare a Python notebook environment

Open this notebook in Antigravity and select a Python kernel from the notebook's kernel/interpreter selector. The exact label may vary by version. This notebook uses only Python's standard library, so no package installation is needed.

Run the code cells from top to bottom. The matching demonstration and tests are offline. The interactive cell asks for a question when you run that cell; you can skip it and use the prewritten example instead.

## 2. Load and validate the embedded FAQ data

The complete JSON dataset is embedded below. The notebook does not read `input/insurance-faq-dataset.json` at runtime, so the notebook remains self-contained. The validation function checks the top-level structure, required fields, non-empty text, allowed optional fields, and unique IDs before the data can be used.

In [ ]:
import json
import re
from difflib import SequenceMatcher

FAQ_JSON = r'''[
  {
    "id": "faq-001",
    "category": "Getting started",
    "question": "What is the Insurance Submission Intake Assistant for?",
    "answer": "It helps users organize submission information, review available documents, summarize supplied details, and identify questions or items that may need follow-up. It supports intake work; it does not make underwriting decisions. Use your insurer’s actual procedures for required steps and approvals."
  },
  {
    "id": "faq-002",
    "category": "Getting started",
    "question": "How should I begin reviewing a submission?",
    "answer": "Start with the submission identifier and the information available to you. Review the supplied application and supporting materials, note what is present or unclear, and follow your insurer’s intake procedure. The correct sequence may vary by insurer and line of business."
  },
  {
    "id": "faq-003",
    "category": "Getting started",
    "question": "Can I ask the assistant questions in everyday language?",
    "answer": "Yes. Ask a clear question about the submission information you are authorized to use, such as “Summarize the documents provided for synthetic submission SYNTH-1001.” Check the answer against its cited source material and your insurer’s procedures."
  },
  {
    "id": "faq-004",
    "category": "Getting started",
    "question": "What should I do if I do not know a submission number?",
    "answer": "Use the search options available in your insurer’s system, such as a business name or another supported identifier. Search fields and access depend on the configured system. Do not use information you are not authorized to access."
  },
  {
    "id": "faq-005",
    "category": "Insurance submission intake",
    "question": "What does submission intake mean?",
    "answer": "Submission intake is the process of receiving and organizing information for review. It may include checking identifiers, documents, and unanswered questions. The exact intake steps and responsibilities depend on the insurer’s procedures."
  },
  {
    "id": "faq-006",
    "category": "Insurance submission intake",
    "question": "What information is useful for identifying a submission?",
    "answer": "A submission ID is often a useful starting point. Depending on the insurer’s system, other supported search details may include a business name or line of business. Confirm the available fields and avoid assuming that a search covers every source."
  },
  {
    "id": "faq-007",
    "category": "Insurance submission intake",
    "question": "Can the assistant summarize a submission?",
    "answer": "It can help summarize information that is supplied or available through an authorized, configured source. Treat the summary as a starting point, check important details against the original materials, and follow your insurer’s review procedure. If no source is connected, the assistant cannot retrieve submission data."
  },
  {
    "id": "faq-008",
    "category": "Insurance submission intake",
    "question": "How should I handle conflicting details in submission documents?",
    "answer": "Record the conflicting values and where each one appears, then ask an authorized underwriter or intake lead how to proceed. Do not silently choose one value. The insurer’s procedures determine how the discrepancy should be resolved."
  },
  {
    "id": "faq-009",
    "category": "Insurance submission intake",
    "question": "Can the assistant decide whether a submission is ready for underwriting?",
    "answer": "No. It may help organize information or show an administrative status when the system supports it, but readiness definitions depend on the insurer’s actual workflow. Administrative completeness is not an underwriting approval or coverage decision."
  },
  {
    "id": "faq-010",
    "category": "Insurance submission intake",
    "question": "Can the assistant draft a broker follow-up?",
    "answer": "It may help prepare a draft based on items you have verified. Review and edit the message before using it. Sending is a separate action and must only occur through an authorized, configured process with the required human confirmation."
  },
  {
    "id": "faq-011",
    "category": "Required documents and missing information",
    "question": "Which documents are required for a submission?",
    "answer": "Requirements vary by insurer, product, line of business, and submission circumstances. Check the insurer’s current checklist or procedures. The assistant should not invent a required-document list."
  },
  {
    "id": "faq-012",
    "category": "Required documents and missing information",
    "question": "How can I tell whether a document is missing?",
    "answer": "Check the applicable insurer-approved checklist and the sources you are authorized to review. If a source was not available or could not be checked, say that the document was not found in the checked sources rather than declaring it missing."
  },
  {
    "id": "faq-013",
    "category": "Required documents and missing information",
    "question": "What is the difference between “not found” and “missing”?",
    "answer": "“Not found” means the item was not located in the sources that were checked. “Missing” should be used only when the applicable process and evidence support that conclusion. Your insurer’s checklist and source coverage determine how to label the status."
  },
  {
    "id": "faq-014",
    "category": "Required documents and missing information",
    "question": "What if I cannot access the document repository?",
    "answer": "State that the source could not be accessed and that the document status is unverified. Follow your insurer’s procedure to restore access or ask an authorized colleague to check. Do not tell a broker that a document is missing based only on an access problem."
  },
  {
    "id": "faq-015",
    "category": "Required documents and missing information",
    "question": "What if a document is present but some details need checking?",
    "answer": "Mark the detail for human verification and point to the relevant source if available. Do not treat an unverified value as confirmed. Follow the insurer’s review process for corrections or clarification."
  },
  {
    "id": "faq-016",
    "category": "Required documents and missing information",
    "question": "Can the assistant decide which checklist applies?",
    "answer": "Only if the insurer has configured and approved that capability and the relevant information is available. Otherwise, confirm the applicable checklist with an authorized intake or underwriting contact."
  },
  {
    "id": "faq-017",
    "category": "Submission status and workflow",
    "question": "Where can I check a submission’s status?",
    "answer": "Use the insurer’s designated system or workflow, if available to you. Status names and meanings are organization-specific. If the assistant cannot access a verified status source, it should say that the status is unavailable rather than guess."
  },
  {
    "id": "faq-018",
    "category": "Submission status and workflow",
    "question": "What does “pending” mean?",
    "answer": "The meaning of “pending” depends on the insurer’s workflow and status definitions. Check the system’s guidance or ask the workflow owner before interpreting it as a particular action or decision."
  },
  {
    "id": "faq-019",
    "category": "Submission status and workflow",
    "question": "What should I do when a search returns no results?",
    "answer": "Check the identifier for typing errors and try another supported search field if permitted. A no-result message means no match was found in the sources searched; it does not prove that no submission exists elsewhere. Follow your insurer’s process if you still cannot locate it."
  },
  {
    "id": "faq-020",
    "category": "Submission status and workflow",
    "question": "Can the assistant show my assigned or priority submissions?",
    "answer": "Only if the system has a configured, authorized, and sufficiently current source for assignment or priority information. If those definitions or data are unavailable, do not treat an example or stale display as a live queue."
  },
  {
    "id": "faq-021",
    "category": "Underwriter review and escalation",
    "question": "Who makes the underwriting decision?",
    "answer": "An authorized underwriting professional makes underwriting decisions under the insurer’s procedures and authority. The assistant can organize or summarize information but must not approve, decline, price, or bind a risk."
  },
  {
    "id": "faq-022",
    "category": "Underwriter review and escalation",
    "question": "When should I escalate a submission question?",
    "answer": "Escalate when the information is conflicting, unclear, unavailable, outside your role, or requires a business decision. Use the insurer’s escalation path and consult the relevant underwriter, manager, or workflow owner."
  },
  {
    "id": "faq-023",
    "category": "Underwriter review and escalation",
    "question": "Should I rely on an AI-generated summary without checking the source?",
    "answer": "No. Treat it as a navigation and review aid. Check important facts against the original documents or records, especially when information is incomplete or inconsistent. Follow your insurer’s procedures for documenting review."
  },
  {
    "id": "faq-024",
    "category": "Data privacy and safe AI usage",
    "question": "Can I enter real customer or submission data into the assistant?",
    "answer": "Only if your insurer has approved this assistant, its data handling, and the specific information for that use. Otherwise, use synthetic examples and do not enter customer, broker, policy, or submission details. Follow your organization’s privacy and security instructions."
  },
  {
    "id": "faq-025",
    "category": "Data privacy and safe AI usage",
    "question": "How should I handle an answer that seems wrong or unsupported?",
    "answer": "Do not rely on it as confirmed. Verify it against an authorized source, correct or report the issue through your insurer’s approved process, and escalate consequential discrepancies to an authorized human reviewer. Never treat an unsupported AI response as an underwriting decision."
  }
]'''

REQUIRED_FIELDS = {"id", "category", "question", "answer"}
ALLOWED_FIELDS = REQUIRED_FIELDS | {"link_text", "url"}


def validate_faq_entries(entries):
    """Check FAQ structure and raise ValueError with a useful explanation."""
    if not isinstance(entries, list):
        raise ValueError("FAQ data must be a JSON array of entry objects.")
    if not entries:
        raise ValueError("FAQ data must contain at least one entry.")

    seen_ids = set()
    for index, entry in enumerate(entries, start=1):
        if not isinstance(entry, dict):
            raise ValueError(f"FAQ entry {index} must be an object.")
        missing = REQUIRED_FIELDS - entry.keys()
        extra = entry.keys() - ALLOWED_FIELDS
        if missing:
            raise ValueError(f"FAQ entry {index} is missing required fields: {sorted(missing)}.")
        if extra:
            raise ValueError(f"FAQ entry {index} has unsupported fields: {sorted(extra)}.")
        for field in REQUIRED_FIELDS:
            value = entry[field]
            if not isinstance(value, str) or not value.strip():
                raise ValueError(f"FAQ entry {index} field '{field}' must be non-empty text.")
        for field in ("link_text", "url"):
            if field in entry and (not isinstance(entry[field], str) or not entry[field].strip()):
                raise ValueError(f"FAQ entry {index} optional field '{field}' must be non-empty text when included.")
        if entry["id"] in seen_ids:
            raise ValueError(f"FAQ IDs must be unique; duplicate ID: {entry['id']}.")
        seen_ids.add(entry["id"])
    return entries


def parse_and_validate_faq_json(raw_json):
    """Parse JSON text, then validate its FAQ entries."""
    try:
        entries = json.loads(raw_json)
    except (json.JSONDecodeError, TypeError):
        raise ValueError("FAQ content is not valid JSON text.") from None
    return validate_faq_entries(entries)


faq_entries = parse_and_validate_faq_json(FAQ_JSON)
print(f"Loaded and validated {len(faq_entries)} FAQ entries.")

## 3. Match questions offline

This simple matcher compares words and character sequences in the user's question with FAQ questions. It is a basic similarity technique, not semantic understanding and not an LLM. A strong match returns the corresponding answer exactly as written in the FAQ data. A weak match abstains. Close matches ask the user to clarify instead of combining answers.

In [ ]:
STOP_WORDS = {
    "a", "an", "the", "what", "which", "who", "when", "where", "why", "how",
    "do", "does", "did", "i", "you", "we", "can", "could", "should", "would",
    "is", "are", "was", "were", "for", "to", "of", "in", "on", "my", "this",
    "that", "it", "and", "or", "with", "be", "about", "if", "me", "assistant",
}


def normalize(text):
    """Lowercase text and remove common question words before comparison."""
    return [word for word in re.findall(r"[a-z0-9]+", text.lower()) if word not in STOP_WORDS]


def question_similarity(user_question, faq_question):
    user_tokens = set(normalize(user_question))
    faq_tokens = set(normalize(faq_question))
    if not user_tokens or not faq_tokens:
        return 0.0
    overlap = len(user_tokens & faq_tokens) / len(user_tokens | faq_tokens)
    sequence = SequenceMatcher(None, " ".join(sorted(user_tokens)), " ".join(sorted(faq_tokens))).ratio()
    return max(overlap, sequence)


def match_faq(user_question, entries, minimum_score=0.45, ambiguity_margin=0.06):
    """Return a dataset answer, clarification options, or a safe unknown result."""
    if not isinstance(user_question, str) or not user_question.strip():
        raise ValueError("Please enter a non-empty question.")

    validated_entries = validate_faq_entries(entries)
    scored = [
        (question_similarity(user_question, item["question"]), item)
        for item in validated_entries
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    best_score, best_item = scored[0]

    if best_score < minimum_score:
        return {"status": "unknown", "message": "The FAQ dataset does not provide an answer to that question."}

    close_matches = [
        item for score, item in scored
        if score >= minimum_score and best_score - score <= ambiguity_margin
    ]
    if len(close_matches) > 1:
        return {"status": "clarify", "questions": [item["question"] for item in close_matches[:3]]}

    return {"status": "matched", "entry": best_item}


def format_faq_result(result):
    """Format a safe response without adding business facts."""
    if result["status"] == "matched":
        entry = result["entry"]
        lines = [entry["answer"]]
        if "link_text" in entry and "url" in entry:
            lines.append(f"Reference: {entry['link_text']} — {entry['url']}")
        return "\n".join(lines)
    if result["status"] == "clarify":
        options = "\n".join(f"- {question}" for question in result["questions"])
        return "I found a few possible FAQ topics. Which one did you mean?\n" + options
    return result["message"]

## 4. Try a valid question

The sample below uses an exact question from the embedded dataset. The answer should match the dataset entry exactly. Change `sample_question` to try another FAQ.

In [ ]:
sample_question = "Who makes the underwriting decision?"
sample_result = match_faq(sample_question, faq_entries)
print(format_faq_result(sample_result))

## 5. Ask your own question (optional interactive cell)

This cell reads a question from your notebook input. It uses offline matching only and does not call an LLM or external service. If there is no suitable FAQ match, it says the dataset does not answer the question. If there are close alternatives, it asks you to choose a topic.

In [ ]:
user_question = input("Enter an insurance intake FAQ question: ")
try:
    result = match_faq(user_question, faq_entries)
except ValueError as error:
    print(error)
else:
    print(format_faq_result(result))

## 6. Tests for matching and malformed FAQ data

The following checks are runnable offline. They cover a valid question, an unsupported question, empty input, invalid JSON, a missing field, a blank field, and duplicate IDs. These checks do not alter the embedded dataset or the original JSON file.

In [ ]:
def expect_invalid(raw_json, description):
    """Assert that malformed FAQ text is rejected before matching."""
    try:
        parse_and_validate_faq_json(raw_json)
    except ValueError:
        return
    raise AssertionError(f"Malformed data was accepted: {description}")


# Valid question: answer is returned verbatim from the FAQ dataset.
valid_entry = next(item for item in faq_entries if item["question"] == "Who makes the underwriting decision?")
valid_result = match_faq(valid_entry["question"], faq_entries)
assert valid_result["status"] == "matched"
assert valid_result["entry"]["answer"] == valid_entry["answer"]

# Unknown question: do not invent a price or underwriting decision.
unknown_result = match_faq("What premium should I charge for a seaside hotel?", faq_entries)
assert unknown_result["status"] == "unknown"
assert "does not provide an answer" in format_faq_result(unknown_result)

# Empty input: reject before any matching or external call.
try:
    match_faq("   ", faq_entries)
except ValueError as error:
    assert "non-empty" in str(error)
else:
    raise AssertionError("Whitespace-only input should have been rejected.")

# Malformed JSON and structurally invalid FAQ data.
expect_invalid("{not-json", "invalid JSON syntax")
expect_invalid(json.dumps([{"id": "faq-x", "category": "Example", "question": "Question?"}]), "missing answer")
expect_invalid(json.dumps([{"id": "faq-x", "category": "Example", "question": "  ", "answer": "Answer"}]), "blank required value")
expect_invalid(json.dumps([
    {"id": "faq-dup", "category": "Example", "question": "First?", "answer": "First."},
    {"id": "faq-dup", "category": "Example", "question": "Second?", "answer": "Second."},
]), "duplicate ID")

print("All FAQ behavior checks passed when this cell is run.")

## 7. Offline FAQ mode and actual LLM calls are different

**This notebook currently provides offline FAQ matching only.** It compares the user's text with the embedded FAQ questions and returns a stored answer, asks for clarification, or abstains. It requires no API key, network access, or provider setup.

An actual LLM API call would send a request to an external or separately configured model service. It is **not implemented in this notebook** because no provider, endpoint, model, or data-handling approval is confirmed for this project. Do not interpret offline matching as an LLM connection. If an API mode is added later, keep it separate and opt-in, use only approved credentials from environment variables, restrict its business context to this FAQ dataset, and obtain approval before sending any content externally. Never send real customer, broker, policy, or submission information in this prototype.

## Finished

You can edit the question and rerun the matching or interactive cells. The assistant can only return answers present in the embedded FAQ JSON. For information not covered by that dataset, consult an authorized person or the insurer's actual procedures; this notebook cannot retrieve submission status or make an underwriting decision.